# Aula 1 — Fundamentos de fluxos agentic com LLMs

## Caso prático: triagem inicial de incidentes técnicos

Nesta prática, você vai montar um workflow de triagem que combina regras determinísticas e chamadas a LLM com saída estruturada.

### Objetivos de aprendizagem

Ao final deste notebook, você será capaz de:
- diferenciar o que deve ficar em código determinístico e o que pode ser delegado ao modelo;
- definir contratos de saída com validação estruturada;
- recuperar contexto relevante antes da síntese final;
- justificar decisões de escalonamento com base em evidências.

### Entregáveis esperados

1. interpretação estruturada de um ticket com validação;
2. análise final com hipóteses, evidências e próximos passos;
3. trilha de execução (trace) mostrando as etapas do workflow.

```text
Ticket -> interpretação estruturada -> validação
      -> ferramentas controladas (logs, status, runbooks)
      -> recuperação de contexto
      -> síntese final -> validação + decisão de escalonamento
```

### Como usar este material

Para cada seção, siga este ciclo:
1. execute as células na ordem;
2. observe a saída indicada no checkpoint;
3. confirme o critério de sucesso antes de avançar.

## 1. Setup

### Objetivo
Configurar o ambiente para rodar o notebook em modo demonstração ou com LLM real.

### Ação
- modo demonstração: mantenha `USE_DEMO_MODE=true`;
- modo com LLM: configure `.env` com `OPENAI_API_KEY`, `OPENAI_MODEL` e `USE_DEMO_MODE=false`.

### O que observar
A saída das próximas células deve mostrar o modo selecionado e o nome do modelo.

### Critério de sucesso
Você consegue identificar, no output, se está em modo demo ou modo real antes de continuar.

In [23]:
from pathlib import Path
import json
import os
from typing import Literal

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [24]:
load_dotenv()

DATA_DIR = Path("../data") if Path("../data").exists() else Path("data")
USE_DEMO_MODE = os.getenv("USE_DEMO_MODE")
OPENAI_MODEL = os.getenv("OPENAI_MODEL")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

print("Modo demonstração:", USE_DEMO_MODE)
print("Modelo configurado:", OPENAI_MODEL)

Modo demonstração: false
Modelo configurado: gpt-4.1-mini


### Leitura dos dados

Nesta etapa, carregamos os dados do cenário (`tickets`, `logs`, `status` e `runbooks`).

### O que você deve validar agora
- os arquivos JSON são lidos sem erro;
- a variável `tickets` retorna uma lista com incidentes;
- os dados carregados serão a fonte de verdade da prática.

### Checkpoint
Se você chegou até aqui, já tem insumos para executar o workflow sem depender de integrações externas.

In [25]:
def read_json(filename: str):
    return json.loads((DATA_DIR / filename).read_text(encoding="utf-8"))

tickets = read_json("tickets.json")
logs_by_ticket = read_json("logs.json")
status_by_service = read_json("service_status.json")
runbooks = read_json("runbooks.json")

tickets

[{'ticket_id': 'INC-2026-001',
  'title': 'Falhas após deploy no serviço de conciliação',
  'description': 'Após o deploy da versão 2.4.1, o pipeline de pagamentos passou a registrar erros 500 e timeouts no serviço de conciliação.',
  'service': 'conciliation-service',
  'reported_by': 'payments-platform',
  'created_at': '2026-07-04T09:15:00'},
 {'ticket_id': 'INC-2026-002',
  'title': 'Lentidão intermitente em relatórios',
  'description': 'Usuários relatam lentidão para gerar relatórios financeiros. Não há evidência de indisponibilidade total.',
  'service': 'reporting-service',
  'reported_by': 'finance-ops',
  'created_at': '2026-07-04T10:20:00'}]

## 2. Definição do contrato de saída

### Objetivo
Definir estruturas que permitam validar automaticamente as respostas do fluxo.

### Por que isso importa
Formato válido não garante decisão correta, mas evita que o restante do sistema quebre por inconsistência estrutural.

### Critério de sucesso
As classes de contrato devem representar os campos obrigatórios para:
- interpretação inicial do ticket;
- análise final do incidente.

In [26]:
class TicketInterpretation(BaseModel):
    system_affected: str
    incident_type: Literal["timeout", "error_rate", "latency", "unknown"]
    severity: Literal["low", "medium", "high", "critical"]
    summary: str
    missing_information: list[str] = Field(default_factory=list)
    should_escalate: bool
    escalation_reason: str | None = None

class IncidentAnalysis(BaseModel):
    incident_id: str
    summary: str
    hypotheses: list[str]
    evidence: list[str]
    next_steps: list[str]
    confidence: Literal["low", "medium", "high"]
    should_escalate: bool
    escalation_reason: str | None = None


A função abaixo recupera um ticket a partir do ID informado.

### Checkpoint
Teste com `INC-2026-001` e confirme se o dicionário retornado contém título, descrição e serviço.

In [27]:
def get_ticket(ticket_id: str) -> dict:
    for ticket in tickets:
        if ticket["ticket_id"] == ticket_id:
            return ticket
    raise ValueError(f"Ticket não encontrado: {ticket_id}")

ticket = get_ticket("INC-2026-001")
ticket


{'ticket_id': 'INC-2026-001',
 'title': 'Falhas após deploy no serviço de conciliação',
 'description': 'Após o deploy da versão 2.4.1, o pipeline de pagamentos passou a registrar erros 500 e timeouts no serviço de conciliação.',
 'service': 'conciliation-service',
 'reported_by': 'payments-platform',
 'created_at': '2026-07-04T09:15:00'}

## 3. Primeiro degrau: implementação determinística

### Objetivo
Criar uma referência confiável de comportamento antes de envolver o modelo.

### O que fazer
Execute a função de interpretação demo e observe como ela classifica severidade, tipo de incidente e decisão de escalonamento.

### O que observar
- a saída já respeita o contrato;
- a lógica é previsível e auditável;
- você consegue explicar por que houve (ou não) escalonamento.

### Critério de sucesso
A interpretação demo retorna um objeto válido de `TicketInterpretation`.

In [28]:
def interpret_ticket_demo(ticket: dict) -> TicketInterpretation:
    description = ticket["description"].lower()

    if "timeout" in description or "erro 500" in description or "erros 500" in description:
        return TicketInterpretation(
            system_affected=ticket["service"],
            incident_type="timeout",
            severity="high",
            summary="Falhas após deploy com timeouts em dependência e erros 500 no serviço.",
            missing_information=[
                "métrica de latência da dependência",
                "impacto em transações concluídas"
            ],
            should_escalate=True,
            escalation_reason="Erros 500 e possível impacto em pagamentos."
        )

    return TicketInterpretation(
        system_affected=ticket["service"],
        incident_type="latency",
        severity="medium",
        summary="Lentidão reportada sem evidência de indisponibilidade total.",
        missing_information=[
            "número de usuários afetados",
            "métrica de latência por rota"
        ],
        should_escalate=False
    )

interpretation = interpret_ticket_demo(ticket)
interpretation


TicketInterpretation(system_affected='conciliation-service', incident_type='timeout', severity='high', summary='Falhas após deploy com timeouts em dependência e erros 500 no serviço.', missing_information=['métrica de latência da dependência', 'impacto em transações concluídas'], should_escalate=True, escalation_reason='Erros 500 e possível impacto em pagamentos.')

## 4. Segundo degrau: mesmo contrato com SDK oficial da OpenAI

### Objetivo
Trocar o componente de interpretação sem alterar o contrato do restante do fluxo.

### Ponto-chave
A mudança é de implementação, não de interface: o schema continua o mesmo.

### Como executar
- sem API ou em demo: o notebook mantém fallback determinístico;
- com API: o modelo preenche a estrutura definida no contrato.

### Critério de sucesso
Você obtém uma interpretação compatível com `TicketInterpretation` em ambos os modos.

### Comparação prática: saída livre vs saída estruturada

1. execute primeiro a versão sem schema e observe a variação do texto;
2. execute depois a versão com parse estruturado;
3. compare previsibilidade e facilidade de validação.

### Autoavaliação
Você consegue dizer claramente o que melhora quando força saída estruturada?

In [30]:
def interpret_ticket_openai(ticket: dict) -> TicketInterpretation:
    if USE_DEMO_MODE == True or not os.getenv("OPENAI_API_KEY"):
        return interpret_ticket_demo(ticket)

    from openai import OpenAI
    print("Interpretando ticket com OpenAI:", ticket["ticket_id"])
    client = OpenAI()
    prompt = f"""
        Você é um componente de triagem de incidentes.

        Interprete o ticket abaixo. Não proponha ações destrutivas.
        Não invente informações ausentes.
        Retorne uma estrutura compatível com o schema fornecido.

    TICKET:
    {json.dumps(ticket, ensure_ascii=False)}
    """

    response = client.responses.create(
        model=OPENAI_MODEL,
        input=prompt,
    )
    return response

response = interpret_ticket_openai(ticket)
response


Interpretando ticket com OpenAI: INC-2026-001


Response(id='resp_0df8cef62cb97bb3006a5eb6aa735081a29858fb2cf47cbf99', created_at=1784592042.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4.1-mini-2025-04-14', object='response', output=[ResponseOutputMessage(id='msg_0df8cef62cb97bb3006a5eb6aace7081a2a6519b90d105d287', content=[ResponseOutputText(annotations=[], text='{\n  "ticket_id": "INC-2026-001",\n  "summary": "Erros 500 e timeouts no serviço de conciliação após deploy da versão 2.4.1",\n  "service": "conciliation-service",\n  "issue_type": "Erro de aplicação",\n  "impact": "Interrupção parcial do pipeline de pagamentos",\n  "symptoms": [\n    "Erros HTTP 500",\n    "Timeouts no serviço de conciliação"\n  ],\n  "reported_by": "payments-platform",\n  "created_at": "2026-07-04T09:15:00",\n  "notes": "Problemas iniciaram após deploy da versão 2.4.1 do serviço"\n}', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None)], parallel_tool_calls=True, temp

In [31]:
print(response.output[0].content[0].text)

{
  "ticket_id": "INC-2026-001",
  "summary": "Erros 500 e timeouts no serviço de conciliação após deploy da versão 2.4.1",
  "service": "conciliation-service",
  "issue_type": "Erro de aplicação",
  "impact": "Interrupção parcial do pipeline de pagamentos",
  "symptoms": [
    "Erros HTTP 500",
    "Timeouts no serviço de conciliação"
  ],
  "reported_by": "payments-platform",
  "created_at": "2026-07-04T09:15:00",
  "notes": "Problemas iniciaram após deploy da versão 2.4.1 do serviço"
}


In [32]:
def interpret_ticket_openai(ticket: dict) -> TicketInterpretation:
    if USE_DEMO_MODE == True or not os.getenv("OPENAI_API_KEY"):
        return interpret_ticket_demo(ticket)

    from openai import OpenAI
    print("Interpretando ticket com OpenAI:", ticket["ticket_id"])
    client = OpenAI()
    prompt = f"""
        Você é um componente de triagem de incidentes.

        Interprete o ticket abaixo. Não proponha ações destrutivas.
        Não invente informações ausentes.
        Retorne uma estrutura compatível com o schema fornecido.

    TICKET:
    {json.dumps(ticket, ensure_ascii=False)}
    """

    response = client.responses.parse(
        model=OPENAI_MODEL,
        input=prompt,
        text_format=TicketInterpretation,
    )
    return response.output_parsed, response

interpretation, response = interpret_ticket_openai(ticket)
interpretation


Interpretando ticket com OpenAI: INC-2026-001


TicketInterpretation(system_affected='conciliation-service', incident_type='error_rate', severity='high', summary='Após o deploy da versão 2.4.1, o serviço de conciliação passou a registrar erros 500 e timeouts, afetando o pipeline de pagamentos.', missing_information=['impacto detalhado no negócio', 'logs específicos do erro', 'passos para reproduzir o problema', 'status atual do serviço', 'ações já tomadas'], should_escalate=True, escalation_reason='Erro crítico pós-deploy afetando pagamentos com erros 500 e timeouts no serviço de conciliação.')

In [33]:
print(response.output[0].content[0].text)

{"system_affected":"conciliation-service","incident_type":"error_rate","severity":"high","summary":"Após o deploy da versão 2.4.1, o serviço de conciliação passou a registrar erros 500 e timeouts, afetando o pipeline de pagamentos.","missing_information":["impacto detalhado no negócio","logs específicos do erro","passos para reproduzir o problema","status atual do serviço","ações já tomadas"],"should_escalate":true,"escalation_reason":"Erro crítico pós-deploy afetando pagamentos com erros 500 e timeouts no serviço de conciliação."}


### Checkpoint de entendimento

Responda antes de seguir:
1. o que você ganha ao usar schema estruturado?
2. qual risco ainda existe mesmo com JSON válido?
3. em que etapa o modelo deve decidir e em que etapa o código deve decidir?

### Resposta esperada (resumo)
- schema melhora previsibilidade e integração;
- regra de domínio ainda precisa de validação determinística;
- modelo sugere e estrutura, código aplica políticas e bloqueios.

## 5. Validações determinísticas

### Objetivo
Aplicar regras de domínio que o schema sozinho não garante.

### Exemplo
Mesmo com JSON válido, uma decisão pode ser inadequada (por exemplo, alta severidade sem escalonamento).

### Critério de sucesso
As validações bloqueiam saídas estruturalmente válidas, porém inconsistentes com a política do domínio.

In [34]:
def validate_interpretation(interpretation: TicketInterpretation) -> TicketInterpretation:
    if interpretation.severity in {"high", "critical"} and not interpretation.should_escalate:
        raise ValueError("Incidentes de alta criticidade devem ser escalados.")

    if not interpretation.system_affected:
        raise ValueError("O sistema afetado não pode estar vazio.")

    return interpretation

validate_interpretation(interpretation)


TicketInterpretation(system_affected='conciliation-service', incident_type='error_rate', severity='high', summary='Após o deploy da versão 2.4.1, o serviço de conciliação passou a registrar erros 500 e timeouts, afetando o pipeline de pagamentos.', missing_information=['impacto detalhado no negócio', 'logs específicos do erro', 'passos para reproduzir o problema', 'status atual do serviço', 'ações já tomadas'], should_escalate=True, escalation_reason='Erro crítico pós-deploy afetando pagamentos com erros 500 e timeouts no serviço de conciliação.')

## 6. Ferramentas controladas

### Objetivo
Enriquecer o contexto do fluxo sem dar acesso irrestrito ao ambiente.

### O que você deve observar
- cada ferramenta tem entrada e saída explícitas;
- o workflow controla quais funções podem ser chamadas;
- os dados retornados são suficientes para análise, sem expor capacidades perigosas.

### Checkpoint
Após executar as células, confirme que logs e status foram recuperados apenas pelas funções permitidas.

In [35]:
def get_logs(ticket_id: str) -> list[str]:
    return logs_by_ticket.get(ticket_id, [])

def get_service_status(service: str) -> dict:
    return status_by_service.get(service, {"status": "unknown"})

def get_runbooks() -> list[dict]:
    return runbooks

ticket_logs = get_logs(ticket["ticket_id"])
service_status = get_service_status(ticket["service"])

ticket_logs, service_status


(['2026-07-04T09:06:12Z INFO deployment version=2.4.1 service=conciliation-service',
  '2026-07-04T09:08:47Z WARN downstream_timeout dependency=ledger-api timeout_ms=3000',
  '2026-07-04T09:09:01Z ERROR http_status=500 endpoint=/reconcile request_id=req-9af',
  '2026-07-04T09:10:15Z ERROR retry_exhausted dependency=ledger-api attempts=3'],
 {'status': 'degraded', 'version': '2.4.1', 'error_rate': 0.18})

## 7. Recuperação seletiva de contexto

### Objetivo
Selecionar runbooks mais relevantes antes da síntese.

### Por que usar essa abordagem
O exemplo usa TF-IDF por ser transparente e fácil de inspecionar. O princípio é o mesmo de recuperação para RAG.

### Critério de sucesso
Você consegue justificar por que os runbooks retornados foram selecionados para o ticket atual.

In [36]:
def retrieve_relevant_runbooks(query: str, runbooks: list[dict], top_k: int = 2) -> list[dict]:
    corpus = [
        f"{runbook['title']} {' '.join(runbook['tags'])} {runbook['content']}"
        for runbook in runbooks
    ]

    vectorizer = TfidfVectorizer()
    matrix = vectorizer.fit_transform(corpus + [query])
    scores = cosine_similarity(matrix[-1], matrix[:-1]).flatten()
    ranked_indices = scores.argsort()[::-1][:top_k]

    return [
        {**runbooks[index], "retrieval_score": round(float(scores[index]), 3)}
        for index in ranked_indices
    ]

query = " ".join([
    ticket["title"],
    ticket["description"],
    interpretation.incident_type,
    interpretation.system_affected,
])

relevant_runbooks = retrieve_relevant_runbooks(query, get_runbooks(), top_k=2)
relevant_runbooks


[{'runbook_id': 'RB-LEDGER-TIMEOUT',
  'title': 'Timeouts na integração com Ledger API',
  'tags': ['ledger-api', 'timeout', 'conciliation'],
  'content': 'Quando houver timeouts repetidos para ledger-api após deploy, verificar compatibilidade de payload, latência da dependência e configuração de timeout. Não realizar rollback automático; escalar para on-call de pagamentos se houver aumento de erros 500 acima de 10%.',
  'retrieval_score': 0.254},
 {'runbook_id': 'RB-GENERAL-INCIDENT',
  'title': 'Triagem inicial de incidentes',
  'tags': ['incident', 'triage', 'severity'],
  'content': 'Incidentes com aumento de erro, indisponibilidade ou impacto financeiro devem registrar evidências, hipótese inicial, próximos passos e decisão de escalonamento. A resposta inicial não deve executar alterações automáticas.',
  'retrieval_score': 0.093}]

## 8. Síntese: primeiro determinística, depois com LLM

### Objetivo
Produzir uma análise inicial que seja útil, rastreável e fundamentada.

### Regras de qualidade
- use apenas evidências disponíveis;
- não invente fatos;
- quando houver incerteza relevante, recomende escalonamento humano.

### Checkpoint
A análise final deve conter: resumo, hipóteses, evidências, próximos passos e confiança.

In [37]:
def synthesize_analysis_demo(
    ticket: dict,
    interpretation: TicketInterpretation,
    evidence: list[str],
    runbooks: list[dict],
) -> IncidentAnalysis:
    return IncidentAnalysis(
        incident_id=ticket["ticket_id"],
        summary=f"Incidente em {interpretation.system_affected}: {interpretation.summary}",
        hypotheses=[
            "A versão recém-deployada pode ter alterado a integração com uma dependência.",
            "A dependência externa pode estar excedendo o timeout configurado."
        ],
        evidence=evidence[:4] + [f"Runbook: {item['title']}" for item in runbooks],
        next_steps=[
            "Verificar métricas e logs da dependência identificada.",
            "Comparar alterações da versão atual com a anterior.",
            "Acionar revisão humana antes de qualquer alteração em produção."
        ],
        confidence="medium",
        should_escalate=interpretation.should_escalate,
        escalation_reason=interpretation.escalation_reason
    )


In [38]:
def synthesize_analysis_openai(
    ticket: dict,
    interpretation: TicketInterpretation,
    evidence: list[str],
    runbooks: list[dict],
) -> IncidentAnalysis:
    if USE_DEMO_MODE or not os.getenv("OPENAI_API_KEY"):
        return synthesize_analysis_demo(ticket, interpretation, evidence, runbooks)

    from openai import OpenAI

    client = OpenAI()
    payload = {
        "ticket": ticket,
        "interpretation": interpretation.model_dump(),
        "evidence": evidence,
        "runbooks": runbooks,
    }

    prompt = f"""
Você produz uma análise inicial de incidentes.

Use somente as evidências e os runbooks fornecidos.
Não invente fatos. Não execute ações. Quando houver risco ou evidência insuficiente, recomende escalonamento humano.
Retorne uma estrutura compatível com o schema solicitado.

DADOS:
{json.dumps(payload, ensure_ascii=False)}
"""

    response = client.responses.parse(
        model=OPENAI_MODEL,
        input=prompt,
        text_format=IncidentAnalysis,
    )
    return response.output_parsed

evidence = ticket_logs + [f"Service status: {service_status}"]
analysis = synthesize_analysis_openai(ticket, interpretation, evidence, relevant_runbooks)
analysis


IncidentAnalysis(incident_id='INC-2026-001', summary='Incidente em conciliation-service: Após o deploy da versão 2.4.1, o serviço de conciliação passou a registrar erros 500 e timeouts, afetando o pipeline de pagamentos.', hypotheses=['A versão recém-deployada pode ter alterado a integração com uma dependência.', 'A dependência externa pode estar excedendo o timeout configurado.'], evidence=['2026-07-04T09:06:12Z INFO deployment version=2.4.1 service=conciliation-service', '2026-07-04T09:08:47Z WARN downstream_timeout dependency=ledger-api timeout_ms=3000', '2026-07-04T09:09:01Z ERROR http_status=500 endpoint=/reconcile request_id=req-9af', '2026-07-04T09:10:15Z ERROR retry_exhausted dependency=ledger-api attempts=3', 'Runbook: Timeouts na integração com Ledger API', 'Runbook: Triagem inicial de incidentes'], next_steps=['Verificar métricas e logs da dependência identificada.', 'Comparar alterações da versão atual com a anterior.', 'Acionar revisão humana antes de qualquer alteração em

## 9. Utilizando LangChain em um fluxo básico

### Quando usar
Use esta seção se você quer comparar a implementação com SDK oficial versus abstração de framework.

### O que não muda
Mesmo com framework, continuam essenciais:
- contratos explícitos;
- validações determinísticas;
- controle de permissões das ferramentas.

### Critério de sucesso
A interpretação com LangChain permanece compatível com o mesmo contrato da seção anterior.

In [39]:
def interpret_ticket_langchain(ticket: dict) -> TicketInterpretation:
    if USE_DEMO_MODE or not os.getenv("OPENAI_API_KEY"):
        return interpret_ticket_demo(ticket)

    from langchain_openai import ChatOpenAI
    from langchain_core.prompts import ChatPromptTemplate

    prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            "Você é um componente de triagem de incidentes. "
            "Não proponha ações destrutivas e não invente informações ausentes."
        ),
        ("human", "Interprete o ticket abaixo:\n{ticket_json}")
    ])

    model = ChatOpenAI(model=OPENAI_MODEL, temperature=0)
    chain = prompt | model.with_structured_output(TicketInterpretation)

    return chain.invoke({
        "ticket_json": json.dumps(ticket, ensure_ascii=False)
    })

# Opcional: compare a saída LangChain com a saída do SDK oficial.
interpretation_langchain = interpret_ticket_langchain(ticket)
interpretation_langchain


TicketInterpretation(system_affected='conciliation-service', incident_type='timeout', severity='high', summary='Falhas após deploy com timeouts em dependência e erros 500 no serviço.', missing_information=['métrica de latência da dependência', 'impacto em transações concluídas'], should_escalate=True, escalation_reason='Erros 500 e possível impacto em pagamentos.')

## 10. Validação final e workflow completo

### Objetivo
Executar o fluxo de ponta a ponta e verificar se cada etapa cumpre sua responsabilidade.

### O que validar na saída final
- `interpretation` válida;
- `analysis` validada;
- `trace` com etapas de carregamento, interpretação, recuperação e validação.

### Critério de sucesso
Você consegue explicar cada etapa do `trace` e apontar onde políticas determinísticas atuam.

In [40]:
def validate_analysis(analysis: IncidentAnalysis) -> IncidentAnalysis:
    if not analysis.evidence:
        raise ValueError("A análise precisa conter evidências.")
    if not analysis.next_steps:
        raise ValueError("A análise precisa ter próximos passos.")
    if analysis.confidence == "low" and not analysis.should_escalate:
        raise ValueError("Baixa confiança deve acionar escalonamento.")
    return analysis

def run_incident_workflow(ticket_id: str, engine: str = "sdk") -> dict:
    trace = []

    ticket = get_ticket(ticket_id)
    trace.append({"step": "ticket_loaded", "ticket_id": ticket_id})

    if engine == "langchain":
        interpretation = interpret_ticket_langchain(ticket)
    else:
        interp_result = interpret_ticket_openai(ticket)
        # interpret_ticket_openai may return either a TicketInterpretation or (TicketInterpretation, response)
        if isinstance(interp_result, tuple):
            interpretation = interp_result[0]
        else:
            interpretation = interp_result

    validate_interpretation(interpretation)
    trace.append({
        "step": "ticket_interpreted",
        "severity": interpretation.severity,
        "should_escalate": interpretation.should_escalate,
        "engine": engine
    })

    logs = get_logs(ticket_id)
    status = get_service_status(ticket["service"])

    query = " ".join([
        ticket["title"],
        ticket["description"],
        interpretation.incident_type,
        interpretation.system_affected,
    ])
    selected_runbooks = retrieve_relevant_runbooks(query, get_runbooks(), top_k=2)
    trace.append({
        "step": "context_retrieved",
        "runbooks": [item["runbook_id"] for item in selected_runbooks]
    })

    evidence = logs + [f"Service status: {status}"]
    analysis = synthesize_analysis_openai(ticket, interpretation, evidence, selected_runbooks)
    validate_analysis(analysis)
    trace.append({
        "step": "analysis_validated",
        "should_escalate": analysis.should_escalate
    })

    return {
        "ticket": ticket,
        "interpretation": interpretation.model_dump(),
        "analysis": analysis.model_dump(),
        "trace": trace
    }

result = run_incident_workflow("INC-2026-001", engine="sdk")
result


Interpretando ticket com OpenAI: INC-2026-001


{'ticket': {'ticket_id': 'INC-2026-001',
  'title': 'Falhas após deploy no serviço de conciliação',
  'description': 'Após o deploy da versão 2.4.1, o pipeline de pagamentos passou a registrar erros 500 e timeouts no serviço de conciliação.',
  'service': 'conciliation-service',
  'reported_by': 'payments-platform',
  'created_at': '2026-07-04T09:15:00'},
 'interpretation': {'system_affected': 'conciliation-service',
  'incident_type': 'error_rate',
  'severity': 'high',
  'summary': 'Após o deploy da versão 2.4.1, o pipeline de pagamentos registra erros 500 e timeouts no serviço de conciliação.',
  'missing_information': ['Impacto no negócio',
   'Número de ocorrências',
   'Logs detalhados',
   'Tempo total de indisponibilidade',
   'Se houve rollback',
   'Solução tentativa'],
  'should_escalate': True,
  'escalation_reason': 'Erro 500 e timeouts impactam diretamente a funcionalidade do pipeline de pagamentos, podendo comprometer operações financeiras críticas.'},
 'analysis': {'inc

### Checklist final do aluno

Antes de encerrar, confirme:
- consigo explicar o papel do contrato estruturado;
- consigo mostrar onde entram validações determinísticas;
- consigo justificar a decisão de escalonamento com evidências;
- consigo diferenciar o que é responsabilidade do modelo e do código.

Se algum item ficou pendente, retorne à seção correspondente e repita o checkpoint.